# Berechnung von Distanzmatrizen für die Feature-Gruppen NAICS, HS und APP

## 0. Setup

In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd

REPO_ROOT = Path.cwd().parent.resolve()
sys.path.insert(0, str(REPO_ROOT))

from src.naics import pairwise_naics_dist_slim
from src.hs import pairwise_hs_dist_slim
from src.application import pairwise_app_dist_slim
from src.parameter_grid import (
    grid_compute_naics_matrices,
    grid_compute_hs_matrices,
    grid_compute_app_matrices,
)
from src.run_parameter_grid.run_grid_search_NAICS import NAICS_PARAM_GRID
from src.run_parameter_grid.run_grid_search_HS import HS_PARAM_GRID
from src.run_parameter_grid.run_grid_search_APP import APP_PARAM_GRID
from sklearn.model_selection import ParameterGrid
from src.paths import DATA, GRID_OUTPUT, DF_DATA

## 1. Daten laden und prüfen

In [2]:
df = pd.read_pickle(DF_DATA)

print(f"Kunden:  {df.shape[0]}")
print(f"Spalten: {df.shape[1]}")
df.head()

Kunden:  2090
Spalten: 23


,primary_naics_code,primary_naics_desc,secondary_naics_code,secondary_naics_desc,primary_naics_code_2d,primary_naics_desc_2d,primary_naics_code_3d,primary_naics_desc_3d,primary_naics_code_4d,primary_naics_desc_4d,...,application_material_set_h2,employees_on_site,employees_total,employees_year,revenue_on_site,on_site_currency,revenue_total,total_currency,revenue_year,customer_code
0,336350,Motor Vehicle Transmission and Power Train Par...,541715,"Research and Development in the Physical, Engi...",33,Primary Metal Manufacturing,336,Transportation Equipment Manufacturing,3363,Motor Vehicle Parts Manufacturing,...,"{1.8_2.1.2, 1.5_2.4.5, 4.2_2.4.5, 5.5_2.1.2, 5...",500,1000,2023,115500000,USD,172800000,USD,2023,1168
1,332721,Precision Turned Product Manufacturing,332313,Plate Work Manufacturing,33,Primary Metal Manufacturing,332,Fabricated Metal Product Manufacturing,3327,"Machine Shops; Turned Product; and Screw, Nut,...",...,"{1.8_2.1.9, 6.1_2.6.1, 1.7_2.1.9, 4.2_2.1.7, 3...",33,33,2023,6260000,USD,6260000,USD,2022,1174
2,237990,Other Heavy and Civil Engineering Construction,541330,Engineering Services,23,Construction,237,Heavy and Civil Engineering Construction,2379,Other Heavy and Civil Engineering Construction,...,"{1.4_2.1.5, 1.9_2.2.1, 1.8_2.2.1, 3.3_2.1.9, 1...",35,35,2024,16140000,USD,16140000,USD,2022,1566
3,332322,Sheet Metal Work Manufacturing,332812,"Metal Coating, Engraving (except Jewelry and S...",33,Primary Metal Manufacturing,332,Fabricated Metal Product Manufacturing,3323,Architectural and Structural Metals Manufacturing,...,"{1.8_2.1.2, 3.1_2.1.7, 1.7_2.1.9, 3.3_2.1.9, 1...",66,66,2022,11090000,USD,11090000,USD,2023,1957
4,333248,All Other Industrial Machinery Manufacturing,333241,Food Product Machinery Manufacturing,33,Primary Metal Manufacturing,333,Machinery Manufacturing,3332,Industrial Machinery Manufacturing,...,"{1.8_2.1.2, 1.8_2.2.2, 1.9_2.2.1, 3.3_2.1.9, 1...",80,80,2023,20650000,USD,20650000,USD,2021,2009


### 1.1 Datenqualität

In [3]:
cols = ["primary_naics_code", "secondary_naics_code", "hs6_code", "application_material_set_h2"]

for col in cols:
    missing = df[col].isna().sum()
    pct = missing / len(df) * 100
    print(f"{col:40s}  fehlend: {missing:>5d}  ({pct:.1f} %)")

primary_naics_code                        fehlend:     0  (0.0 %)
secondary_naics_code                      fehlend:    45  (2.2 %)
hs6_code                                  fehlend:     0  (0.0 %)
application_material_set_h2               fehlend:     0  (0.0 %)


## 2. NAICS-Distanzmatrizen

### 2.1 Verfügbare Parameter
Die Funktion `pairwise_naics_dist_slim` unterstützt folgende Parameter:

| Parameter | Optionen | Beschreibung |
|---|---|---|
| `metric` | `hamming`, `w_hamming`, `lcp`, `block_weights` | Distanzmetrik auf 6-stelligen NAICS-Codes |
| `gating` | `True`, `False` | Bei `True`: Distanz = 1.0 wenn Sektor (Stellen 0–1) verschieden |
| `weight_scheme` | `linear`, `exponential`, `poly_convex`, `poly_concave` | Gewichtungsschema, nur relevant für `w_hamming` |
| `alpha_penalty` | `float` (z.B. `0.20`) | Strafterm für Secondary-NAICS-Code-Verwendung |
| `use_primary_only` | `True`, `False` | Bei `True`: nur Primary-Code, Secondary wird ignoriert |
| `normalize` | `True`, `False` | Distanz auf [0, 1] normalisieren |
| `n_digits` | `int` (Standard: `6`) | Anzahl der NAICS-Stellen |
| `output` | `square`, `long` | Ausgabeformat: quadratische Matrix oder Long-Format |
| `return_source` | `True`, `False` | Quellspalte (pp/ps/sp/ss) im Long-Format ergänzen |
| `primary_col` | Spaltenname | Spalte mit Primary-NAICS-Code |
| `secondary_col` | Spaltenname | Spalte mit Secondary-NAICS-Code |
| `id_col` | Spaltenname | Spalte mit Kunden-ID |

### 2.2 Einzelberechnung (Beispiel)

In [4]:
naics_example = pairwise_naics_dist_slim(
    df,
    primary_col="primary_naics_code",
    secondary_col="secondary_naics_code",
    id_col="customer_code",
    metric="w_hamming",
    weight_scheme="linear",
    alpha_penalty=0.15,
    use_primary_only=False,
    gating=True,
    normalize=True,
    output="square",
)

print(f"Shape: {naics_example.shape}")
naics_example.head()

Shape: (2090, 2090)


,1168,1174,1566,1957,2009,1256,1926,317,1663,995,...,96,1870,1161,323,2035,872,1075,1554,1021,79
1168,0.00,0.850000,1.0,0.700000,1.0,1.000000,0.850000,1.0,1.000000,0.90,...,1.000000,0.900000,1.00,0.9,1.0,1.0,1.00,1.000000,1.0,1.000000
1174,0.85,0.000000,1.0,0.666667,1.0,1.000000,0.816667,0.8,0.950000,0.95,...,1.000000,0.666667,0.95,0.8,1.0,1.0,0.95,0.950000,1.0,1.000000
1566,1.00,1.000000,0.0,1.000000,1.0,0.833333,1.000000,1.0,1.000000,1.00,...,1.000000,1.000000,1.00,1.0,1.0,1.0,1.00,1.000000,1.0,1.000000
1957,0.70,0.666667,1.0,0.000000,1.0,1.000000,0.150000,0.8,0.950000,0.95,...,1.000000,0.666667,0.95,0.8,1.0,1.0,0.95,0.950000,1.0,1.000000
2009,1.00,1.000000,1.0,1.000000,0.0,1.000000,1.000000,1.0,0.833333,1.00,...,0.833333,0.816667,1.00,1.0,1.0,1.0,1.00,0.833333,1.0,0.833333


### 2.3 Grid Search

In [5]:
combos = list(ParameterGrid(NAICS_PARAM_GRID))
print(f"Anzahl Kombinationen: {len(combos)}")
pd.DataFrame(combos)

Anzahl Kombinationen: 14


,alpha_penalty,gating,metric,n_digits,normalize,output,use_primary_only,weight_scheme
0,0.15,True,hamming,6,True,square,True,NaN
1,0.15,True,hamming,6,True,square,False,NaN
2,0.15,True,w_hamming,6,True,square,True,linear
3,0.15,True,w_hamming,6,True,square,True,exponential
4,0.15,True,w_hamming,6,True,square,True,poly_convex
5,0.15,True,w_hamming,6,True,square,True,poly_concave
6,0.15,True,w_hamming,6,True,square,False,linear
7,0.15,True,w_hamming,6,True,square,False,exponential
8,0.15,True,w_hamming,6,True,square,False,poly_convex
9,0.15,True,w_hamming,6,True,square,False,poly_concave


In [6]:
%%time

registry_naics = grid_compute_naics_matrices(
    df,
    NAICS_PARAM_GRID,
    primary_col="primary_naics_code",
    secondary_col="secondary_naics_code",
    id_col="customer_code",
    output=GRID_OUTPUT / "NAICS",
)

INFO:src.parameter_grid:Grid hat 14 Kombinationen
INFO:src.parameter_grid:Zielordner: C:\Users\ivan_\PycharmProjects\Praxisprojekt\grid_outputs\NAICS
INFO:src.parameter_grid:[1/14] [naics_000_hamming_a0p15_pp-only_d6_square] {'alpha_penalty': 0.15, 'gating': True, 'metric': 'hamming', 'n_digits': 6, 'normalize': True, 'output': 'square', 'use_primary_only': True}
INFO:src.parameter_grid:[2/14] [naics_001_hamming_a0p15_d6_square] {'alpha_penalty': 0.15, 'gating': True, 'metric': 'hamming', 'n_digits': 6, 'normalize': True, 'output': 'square', 'use_primary_only': False}
INFO:src.parameter_grid:[3/14] [naics_002_w-hamming_linear_a0p15_pp-only_d6_square] {'alpha_penalty': 0.15, 'gating': True, 'metric': 'w_hamming', 'n_digits': 6, 'normalize': True, 'output': 'square', 'use_primary_only': True, 'weight_scheme': 'linear'}
INFO:src.parameter_grid:[4/14] [naics_003_w-hamming_exponential_a0p15_pp-only_d6_square] {'alpha_penalty': 0.15, 'gating': True, 'metric': 'w_hamming', 'n_digits': 6, 'nor

CPU times: total: 5.12 s
Wall time: 2.22 s


In [7]:
print(f"Erzeugte Matrizen: {len(registry_naics)}")
registry_naics

Erzeugte Matrizen: 14


,name,file,alpha_penalty,gating,metric,n_digits,normalize,output,use_primary_only,weight_scheme
0,naics_000_hamming_a0p15_pp-only_d6_square,C:\Users\ivan_\PycharmProjects\Praxisprojekt\g...,0.15,True,hamming,6,True,square,True,NaN
1,naics_001_hamming_a0p15_d6_square,C:\Users\ivan_\PycharmProjects\Praxisprojekt\g...,0.15,True,hamming,6,True,square,False,NaN
2,naics_002_w-hamming_linear_a0p15_pp-only_d6_sq...,C:\Users\ivan_\PycharmProjects\Praxisprojekt\g...,0.15,True,w_hamming,6,True,square,True,linear
3,naics_003_w-hamming_exponential_a0p15_pp-only_...,C:\Users\ivan_\PycharmProjects\Praxisprojekt\g...,0.15,True,w_hamming,6,True,square,True,exponential
4,naics_004_w-hamming_poly-convex_a0p15_pp-only_...,C:\Users\ivan_\PycharmProjects\Praxisprojekt\g...,0.15,True,w_hamming,6,True,square,True,poly_convex
5,naics_005_w-hamming_poly-concave_a0p15_pp-only...,C:\Users\ivan_\PycharmProjects\Praxisprojekt\g...,0.15,True,w_hamming,6,True,square,True,poly_concave
6,naics_006_w-hamming_linear_a0p15_d6_square,C:\Users\ivan_\PycharmProjects\Praxisprojekt\g...,0.15,True,w_hamming,6,True,square,False,linear
7,naics_007_w-hamming_exponential_a0p15_d6_square,C:\Users\ivan_\PycharmProjects\Praxisprojekt\g...,0.15,True,w_hamming,6,True,square,False,exponential
8,naics_008_w-hamming_poly-convex_a0p15_d6_square,C:\Users\ivan_\PycharmProjects\Praxisprojekt\g...,0.15,True,w_hamming,6,True,square,False,poly_convex
9,naics_009_w-hamming_poly-concave_a0p15_d6_square,C:\Users\ivan_\PycharmProjects\Praxisprojekt\g...,0.15,True,w_hamming,6,True,square,False,poly_concave


## 3. HS-Distanzmatrizen

### 3.1 Verfügbare Parameter

Die Funktion `pairwise_hs_dist_slim` unterstützt folgende Parameter:

| Parameter | Optionen | Beschreibung |
|---|---|---|
| `metric` | `jaccard`, `dice`, `overlap`, `jaccard_weighted`, `dice_weighted`, `overlap_weighted`, `cartesian_mean`, `chamfer`, `knn_mean` | Set-Engine oder Punkt-Engine |
| `inner_metric` | `lcp`, `hamming`, `w_hamming`, `block_weights` | Punkt-Distanzfunktion, nur für `cartesian_mean`, `chamfer`, `knn_mean` |
| `weight_scheme` | `linear`, `exponential`, `poly_convex`, `poly_concave` | Nur relevant für `inner_metric=w_hamming` |
| `gating` | `True`, `False` | Erste 2 Ziffern (Kapitel) müssen übereinstimmen, sonst 1.0 |
| `k` | `int` (z.B. `3`) | Anzahl Nachbarn, nur für `knn_mean` |
| `level` | `int` (Standard: `6`) | Anzahl der HS-Stellen |
| `weights` | `dict` oder `None` | Optionale manuelle Gewichtung |
| `output` | `square`, `long` | Ausgabeformat |
| `id_col` | Spaltenname | Spalte mit Kunden-ID |
| `hs_col` | Spaltenname | Spalte mit HS-Code-Sets |

### 3.2 Einzelberechnung (Beispiel)

In [8]:
hs_example = pairwise_hs_dist_slim(
    df,
    id_col="customer_code",
    hs_col="hs6_code",
    metric="chamfer",
    inner_metric="lcp",
    output="square",
)

print(f"Shape: {hs_example.shape}")
hs_example.head()

Shape: (2090, 2090)


,1168,1174,1566,1957,2009,1256,1926,317,1663,995,...,96,1870,1161,323,2035,872,1075,1554,1021,79
1168,0.000000,0.777778,0.627778,1.0,0.777778,1.000000,1.0000,1.000,0.777778,1.0,...,0.805556,0.729167,1.0,1.00,0.833333,1.0,0.861111,1.0,1.0,0.733333
1174,0.777778,0.000000,0.575000,1.0,0.513889,1.000000,1.0000,1.000,0.333333,1.0,...,0.548611,0.604167,1.0,1.00,0.673611,1.0,0.750000,1.0,1.0,0.662500
1566,0.627778,0.575000,0.000000,1.0,0.433333,1.000000,1.0000,1.000,0.566667,1.0,...,0.633333,0.404167,1.0,1.00,0.677778,1.0,0.750000,1.0,1.0,0.466667
1957,1.000000,1.000000,1.000000,0.0,1.000000,0.743056,0.8125,0.875,1.000000,1.0,...,1.000000,1.000000,1.0,0.75,0.902778,1.0,1.000000,1.0,1.0,1.000000
2009,0.777778,0.513889,0.433333,1.0,0.000000,1.000000,1.0000,1.000,0.666667,1.0,...,0.527778,0.395833,1.0,1.00,0.722222,1.0,0.750000,1.0,1.0,0.483333


### 3.3 Grid Search

In [9]:
combos = list(ParameterGrid(HS_PARAM_GRID))
print(f"Anzahl Kombinationen: {len(combos)}")
pd.DataFrame(combos)

Anzahl Kombinationen: 83


,level,metric,output,inner_metric,k,gating,weight_scheme
0,6,jaccard,square,NaN,NaN,NaN,NaN
1,6,overlap,square,NaN,NaN,NaN,NaN
2,6,dice,square,NaN,NaN,NaN,NaN
3,6,jaccard_weighted,square,NaN,NaN,NaN,NaN
4,6,overlap_weighted,square,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...
78,6,knn_mean,square,block_weights,6.0,True,NaN
79,6,knn_mean,square,block_weights,7.0,True,NaN
80,6,knn_mean,square,block_weights,8.0,True,NaN
81,6,knn_mean,square,block_weights,9.0,True,NaN


In [23]:
%%time

registry_hs = grid_compute_hs_matrices(
    df,
    HS_PARAM_GRID,
    id_col="customer_code",
    hs_col="hs6_code",
    output=GRID_OUTPUT / "HS",
)

INFO:src.parameter_grid:Grid hat 83 Kombinationen
INFO:src.parameter_grid:Zielordner: C:\Users\ivan_\PycharmProjects\Praxisprojekt\grid_outputs\HS
INFO:src.parameter_grid:[1/83] [hs_000_jaccard_square] {'level': 6, 'metric': 'jaccard', 'output': 'square'}
INFO:src.parameter_grid:[2/83] [hs_001_overlap_square] {'level': 6, 'metric': 'overlap', 'output': 'square'}
INFO:src.parameter_grid:[3/83] [hs_002_dice_square] {'level': 6, 'metric': 'dice', 'output': 'square'}
INFO:src.parameter_grid:[4/83] [hs_003_jaccard-weighted_square] {'level': 6, 'metric': 'jaccard_weighted', 'output': 'square'}
INFO:src.parameter_grid:[5/83] [hs_004_overlap-weighted_square] {'level': 6, 'metric': 'overlap_weighted', 'output': 'square'}
INFO:src.parameter_grid:[6/83] [hs_005_dice-weighted_square] {'level': 6, 'metric': 'dice_weighted', 'output': 'square'}
INFO:src.parameter_grid:[7/83] [hs_006_cartesian-mean_lcp_square] {'inner_metric': 'lcp', 'level': 6, 'metric': 'cartesian_mean', 'output': 'square'}
INFO:sr

CPU times: total: 3min 47s
Wall time: 1min 58s


### 3.4 Ergebnis-Inspektion

In [24]:
print(f"Erzeugte Matrizen: {len(registry_hs)}")
registry_hs

Erzeugte Matrizen: 83


,name,file,level,metric,output,inner_metric,k,gating,weight_scheme
0,hs_000_jaccard_square,C:\Users\ivan_\PycharmProjects\Praxisprojekt\g...,6,jaccard,square,NaN,NaN,NaN,NaN
1,hs_001_overlap_square,C:\Users\ivan_\PycharmProjects\Praxisprojekt\g...,6,overlap,square,NaN,NaN,NaN,NaN
2,hs_002_dice_square,C:\Users\ivan_\PycharmProjects\Praxisprojekt\g...,6,dice,square,NaN,NaN,NaN,NaN
3,hs_003_jaccard-weighted_square,C:\Users\ivan_\PycharmProjects\Praxisprojekt\g...,6,jaccard_weighted,square,NaN,NaN,NaN,NaN
4,hs_004_overlap-weighted_square,C:\Users\ivan_\PycharmProjects\Praxisprojekt\g...,6,overlap_weighted,square,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...
78,hs_078_knn-mean_block-weights_k6_square,C:\Users\ivan_\PycharmProjects\Praxisprojekt\g...,6,knn_mean,square,block_weights,6.0,True,NaN
79,hs_079_knn-mean_block-weights_k7_square,C:\Users\ivan_\PycharmProjects\Praxisprojekt\g...,6,knn_mean,square,block_weights,7.0,True,NaN
80,hs_080_knn-mean_block-weights_k8_square,C:\Users\ivan_\PycharmProjects\Praxisprojekt\g...,6,knn_mean,square,block_weights,8.0,True,NaN
81,hs_081_knn-mean_block-weights_k9_square,C:\Users\ivan_\PycharmProjects\Praxisprojekt\g...,6,knn_mean,square,block_weights,9.0,True,NaN


## 4. Applikations-Distanzmatrizen

### 4.1 Verfügbare Parameter

Die Funktion `pairwise_app_dist_slim` unterstützt folgende Parameter:

| Parameter | Optionen | Beschreibung |
|---|---|---|
| `metric` | `jaccard`, `dice`, `overlap`, `jaccard_weighted`, `dice_weighted`, `overlap_weighted`, `cartesian_mean`, `chamfer`, `knn_mean` | Set-Engine oder Punkt-Engine |
| `inner_metric` | `exact`, `lcp`, `hamming`, `w_hamming`, `block_weights` | Punkt-Distanzfunktion, nur für Punkt-Engine |
| `weight_scheme` | `linear`, `exponential`, `poly_convex`, `poly_concave` | Nur relevant für `inner_metric=w_hamming` |
| `gating` | `True`, `False` | Erste 2 Ziffern müssen übereinstimmen, sonst 1.0 |
| `k` | `int` (z.B. `3`) | Anzahl Nachbarn, nur für `knn_mean` |
| `product_filter` | `None`, `"0601"`, `"0602"`, `"0604"`, `"0201"` | Filtert auf produktspezifische Applikationen |
| `weights` | `dict` oder `None` | Optionale manuelle Gewichtung |
| `output` | `square`, `long` | Ausgabeformat |
| `id_col` | Spaltenname | Spalte mit Kunden-ID |
| `am_col` | Spaltenname | Spalte mit Applikations-Material-Code-Sets |

### 4.2 Einzelberechnung (Beispiel)

In [9]:
app_example = pairwise_app_dist_slim(
    df,
    id_col="customer_code",
    am_col="application_material_set_h2",
    metric="cartesian_mean",
    inner_metric="w_hamming",
    weight_scheme="exponential",
    gating=True,
    product_filter=None,
    output="square",
)

print(f"Shape: {app_example.shape}")
app_example.head()

Shape: (2090, 2090)


,1168,1174,1566,1957,2009,1256,1926,317,1663,995,...,96,1870,1161,323,2035,872,1075,1554,1021,79
1168,0.000000,0.803571,0.807292,0.756944,0.760417,0.808333,0.824074,0.652778,0.815476,0.840278,...,0.803571,0.765625,0.805556,0.734375,0.755952,0.758333,0.776042,0.822917,0.758333,0.750000
1174,0.803571,0.000000,0.839286,0.761905,0.767857,0.819048,0.857143,0.698413,0.795918,0.841270,...,0.809524,0.833333,0.801587,0.755952,0.863946,0.857143,0.767857,0.803571,0.857143,0.738095
1566,0.807292,0.839286,0.000000,0.798611,0.812500,0.808333,0.865741,0.750000,0.845238,0.847222,...,0.869048,0.786458,0.888889,0.786458,0.779762,0.816667,0.848958,0.817708,0.808333,0.791667
1957,0.756944,0.761905,0.798611,0.000000,0.743056,0.788889,0.833333,0.648148,0.761905,0.759259,...,0.833333,0.784722,0.824074,0.694444,0.793651,0.811111,0.777778,0.784722,0.811111,0.708333
2009,0.760417,0.767857,0.812500,0.743056,0.000000,0.800000,0.833333,0.652778,0.785714,0.826389,...,0.803571,0.796875,0.798611,0.729167,0.803571,0.800000,0.755208,0.802083,0.800000,0.729167


In [10]:
combos = list(ParameterGrid(APP_PARAM_GRID))
print(f"Anzahl Kombinationen: {len(combos)}")
pd.DataFrame(combos)

Anzahl Kombinationen: 470


,metric,output,product_filter,inner_metric,k,gating,weight_scheme
0,jaccard,square,None,NaN,NaN,NaN,NaN
1,jaccard,square,0601,NaN,NaN,NaN,NaN
2,jaccard,square,0602,NaN,NaN,NaN,NaN
3,jaccard,square,0604,NaN,NaN,NaN,NaN
4,jaccard,square,0201,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...
465,knn_mean,square,None,lcp,10.0,NaN,NaN
466,knn_mean,square,0601,lcp,10.0,NaN,NaN
467,knn_mean,square,0602,lcp,10.0,NaN,NaN
468,knn_mean,square,0604,lcp,10.0,NaN,NaN


In [12]:
%%time

registry_app = grid_compute_app_matrices(
    df,
    APP_PARAM_GRID,
    id_col="customer_code",
    am_col="application_material_set_h2",
    output=GRID_OUTPUT / "APP",
)

INFO:src.parameter_grid:Grid hat 470 Kombinationen
INFO:src.parameter_grid:Zielordner: C:\Users\ivan_\PycharmProjects\Praxisprojekt\grid_outputs\APP
INFO:src.parameter_grid:[1/470] [app_000_jaccard_square] {'metric': 'jaccard', 'output': 'square', 'product_filter': None}
INFO:src.parameter_grid:[2/470] [app_001_jaccard_pf0601_square] {'metric': 'jaccard', 'output': 'square', 'product_filter': '0601'}
INFO:src.parameter_grid:[3/470] [app_002_jaccard_pf0602_square] {'metric': 'jaccard', 'output': 'square', 'product_filter': '0602'}
INFO:src.parameter_grid:[4/470] [app_003_jaccard_pf0604_square] {'metric': 'jaccard', 'output': 'square', 'product_filter': '0604'}
INFO:src.parameter_grid:[5/470] [app_004_jaccard_pf0201_square] {'metric': 'jaccard', 'output': 'square', 'product_filter': '0201'}
INFO:src.parameter_grid:[6/470] [app_005_overlap_square] {'metric': 'overlap', 'output': 'square', 'product_filter': None}
INFO:src.parameter_grid:[7/470] [app_006_overlap_pf0601_square] {'metric': 'o

CPU times: total: 18min 28s
Wall time: 8min 15s


### 4.4 Ergebnis-Inspektion

In [21]:
print(f"Erzeugte Matrizen: {len(registry_app)}")
registry_app

Erzeugte Matrizen: 470


,name,file,metric,output,product_filter,inner_metric,k,gating,weight_scheme
0,app_000_jaccard_square,C:\Users\ivan_\PycharmProjects\Praxisprojekt\g...,jaccard,square,None,NaN,NaN,NaN,NaN
1,app_001_jaccard_pf0601_square,C:\Users\ivan_\PycharmProjects\Praxisprojekt\g...,jaccard,square,0601,NaN,NaN,NaN,NaN
2,app_002_jaccard_pf0602_square,C:\Users\ivan_\PycharmProjects\Praxisprojekt\g...,jaccard,square,0602,NaN,NaN,NaN,NaN
3,app_003_jaccard_pf0604_square,C:\Users\ivan_\PycharmProjects\Praxisprojekt\g...,jaccard,square,0604,NaN,NaN,NaN,NaN
4,app_004_jaccard_pf0201_square,C:\Users\ivan_\PycharmProjects\Praxisprojekt\g...,jaccard,square,0201,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...
465,app_465_knn-mean_lcp_k10_square,C:\Users\ivan_\PycharmProjects\Praxisprojekt\g...,knn_mean,square,None,lcp,10.0,NaN,NaN
466,app_466_knn-mean_lcp_k10_pf0601_square,C:\Users\ivan_\PycharmProjects\Praxisprojekt\g...,knn_mean,square,0601,lcp,10.0,NaN,NaN
467,app_467_knn-mean_lcp_k10_pf0602_square,C:\Users\ivan_\PycharmProjects\Praxisprojekt\g...,knn_mean,square,0602,lcp,10.0,NaN,NaN
468,app_468_knn-mean_lcp_k10_pf0604_square,C:\Users\ivan_\PycharmProjects\Praxisprojekt\g...,knn_mean,square,0604,lcp,10.0,NaN,NaN


## 5. Zusammenfassung

In [25]:
print(f"NAICS-Matrizen:  {len(registry_naics)}")
print(f"HS-Matrizen:     {len(registry_hs)}")
print(f"APP-Matrizen:    {len(registry_app)}")
print(f"Gesamt:          {len(registry_naics) + len(registry_hs) + len(registry_app)}")

NAICS-Matrizen:  14
HS-Matrizen:     83
APP-Matrizen:    470
Gesamt:          567
